# Kaggle Training Notebook — ResNet34

Hướng dẫn sử dụng:
1. Upload project và dataset lên Kaggle
2. Import notebook này vào Kaggle
3. Chỉnh sửa **Cell 2** — thay đổi đường dẫn dataset
4. Nhấn **Save Version** — chạy training
5. Kaggle tự động lưu output vào panel Output

**Model:** ResNet34 | 60 epochs | Batch 28 | Image 224

In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN
!pip install albumentations PyYAML tensorboard scikit-learn matplotlib seaborn opencv-python tqdm pillow pandas -q

In [ ]:
# BƯỚC 2: KHAI BÁO ĐƯỜNG DẪN
import os
import sys
from pathlib import Path

# ============================================================
# THAY ĐỔI ĐƯỜNG DẪN PHÙ HỢP VỚI DATASET CỦA BẠN
# ============================================================
PROJECT_ROOT = Path("/kaggle/input/datasets/tranmanh1312/ai-durian-disease-detection")
DATA_ROOT    = Path("/kaggle/input/datasets/tranmanh1312/durian-processed-data/data-leaf")
# ============================================================

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")
print(f"Project root: {PROJECT_ROOT} (exists: {PROJECT_ROOT.exists()})")
print(f"Data root:    {DATA_ROOT}    (exists: {DATA_ROOT.exists()})")

for split in ["train", "val", "test"]:
    p = DATA_ROOT / split
    n = sum(1 for ext in IMG_EXTENSIONS for _ in p.rglob(f"*{ext}")) if p.exists() else 0
    print(f"  {split}: {n} images")

In [ ]:
# BƯỚC 3: IMPORT THƯ VIỆN
import json
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from collections import Counter
from tqdm import tqdm
import cv2
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

In [ ]:
# BƯỚC 4: CẤU HÌNH MODEL
MODEL_NAME = "resnet34"

MODEL_CONFIGS = {
    "resnet34": {
        "image_size": 224,
        "batch_size": 28,
        "num_epochs": 60,
        "lr": 0.00025,
        "dropout": 0.35,
    },
}

cfg = MODEL_CONFIGS[MODEL_NAME]
OUTPUT_DIR = Path(f"/kaggle/working/results/{MODEL_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"===== Training {MODEL_NAME.upper()} =====")
print(f"Image size: {cfg['image_size']}, Batch size: {cfg['batch_size']}, Epochs: {cfg['num_epochs']}, LR: {cfg['lr']}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# BƯỚC 5: THIẾT LẬP SEED & DEVICE
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = getattr(torch.cuda.get_device_properties(0), 'total_memory', None)
    if vram is None:
        vram = torch.cuda.get_device_properties(0).total_mem
    print(f"VRAM: {vram / 1e9:.1f} GB")

In [ ]:
# BƯỚC 6: DATASET & DATALOADER
CLASS_NAMES   = ["ALGAL_LEAF_SPOT", "ALLOCARIDARA_ATTACK", "HEALTHY_LEAF", "LEAF_BLIGHT", "PHOMOPSIS_LEAF_SPOT"]
NUM_CLASSES   = len(CLASS_NAMES)
IMAGE_SIZE    = cfg["image_size"]
MEAN          = [0.485, 0.456, 0.406]
STD           = [0.229, 0.224, 0.225]
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

class DurianLeafDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, class_names, transforms=None, return_paths=False):
        self.root_dir = Path(root_dir)
        self.class_names = list(class_names)
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}
        self.transforms = transforms
        self.return_paths = return_paths
        self.samples = self._gather_samples()

    def _gather_samples(self):
        samples = []
        for cls in self.class_names:
            class_dir = self.root_dir / cls
            if not class_dir.exists():
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in IMG_EXTENSIONS:
                    samples.append((str(img_path), self.class_to_idx[cls]))
        samples.sort(key=lambda x: x[0])
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = np.array(Image.open(image_path).convert("RGB"))
        if self.transforms:
            image_tensor = self.transforms(image=image)["image"]
        sample = {"image": image_tensor, "label": label}
        if self.return_paths:
            sample["path"] = image_path
        return sample

train_transform = A.Compose([
    A.LongestMaxSize(IMAGE_SIZE),
    A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Rotate(limit=30, p=0.8),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), p=0.15),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.LongestMaxSize(IMAGE_SIZE),
    A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

train_dataset = DurianLeafDataset(DATA_ROOT / "train", CLASS_NAMES, train_transform)
val_dataset   = DurianLeafDataset(DATA_ROOT / "val",   CLASS_NAMES, eval_transform,  return_paths=True)
test_dataset  = DurianLeafDataset(DATA_ROOT / "test",  CLASS_NAMES, eval_transform,  return_paths=True)

labels = [l for _, l in train_dataset.samples]
counts = Counter(labels)
total  = sum(counts.values())
weights = {c: total / (NUM_CLASSES * cnt) for c, cnt in counts.items()}
sampler = WeightedRandomSampler([weights[l] for l in labels], len(labels), replacement=True)

train_loader = DataLoader(train_dataset, sampler=sampler, batch_size=cfg["batch_size"], num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   shuffle=False,      batch_size=cfg["batch_size"], num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  shuffle=False,      batch_size=cfg["batch_size"], num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"Val:   {len(val_dataset)} images, Test: {len(test_dataset)} images")
print(f"Class distribution (train): {dict(counts)}")

In [ ]:
# BƯỚC 7: BUILD MODEL
import torchvision.models as models

def create_model(name, num_classes, dropout):
    if name == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    return m.to(DEVICE)

model = create_model(MODEL_NAME, NUM_CLASSES, cfg["dropout"])
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}, Total params: {total_params:,}")

In [ ]:
# BƯỚC 8: TRAINING SETUP
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)

total_epochs = cfg["num_epochs"]
warmup_epochs = 3

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * min(progress, 1.0)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
use_amp = cfg.get("mixed_precision", False) and torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)
print(f"Mixed precision: {use_amp}")

In [ ]:
# BƯỚC 9: TRAINING LOOP
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
writer = SummaryWriter(OUTPUT_DIR / "tensorboard")

for epoch in range(1, total_epochs + 1):
    model.train()
    running_loss, running_acc = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{total_epochs} [Train]")
    for batch in pbar:
        images = batch["image"].to(DEVICE)
        labels_batch = batch["label"].to(DEVICE)
        optimizer.zero_grad()

        with autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = outputs.argmax(dim=1)
        acc = (preds == labels_batch).float().mean().item()
        running_loss += loss.item()
        running_acc += acc
        pbar.set_postfix(loss=running_loss / (pbar.n + 1), acc=running_acc / (pbar.n + 1))

    train_loss = running_loss / len(train_loader)
    train_acc  = running_acc  / len(train_loader)

    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{total_epochs} [Val]", leave=False):
            images = batch["image"].to(DEVICE)
            labels_batch = batch["label"].to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels_batch)
            preds = outputs.argmax(dim=1)
            acc = (preds == labels_batch).float().mean().item()
            val_loss += loss.item()
            val_acc  += acc

    val_loss /= len(val_loader)
    val_acc  /= len(val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    writer.add_scalars("Loss",     {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("Accuracy",  {"train": train_acc,  "val": val_acc},  epoch)
    writer.add_scalar("LR", current_lr, epoch)

    print(f"Epoch {epoch:3d}/{total_epochs} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
          f"LR={current_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_metric": val_loss,
            "history": history,
        }, OUTPUT_DIR / "best_model.pth")
        print(f"    [Saved] val_loss={val_loss:.4f}")

writer.close()

with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print(f"\n===== Training Complete =====")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Results saved to: {OUTPUT_DIR}")

In [ ]:
# BƯỚC 10: EVALUATE TRÊN TEST SET
checkpoint = torch.load(OUTPUT_DIR / "best_model.pth", map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating on test set"):
        images = batch["image"].to(DEVICE)
        labels_batch = batch["label"].to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4)

print(f"\n===== Test Set Results =====")
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

torch.save({
    "test_accuracy": accuracy,
    "confusion_matrix": cm,
    "classification_report": report,
    "all_preds": all_preds,
    "all_labels": all_labels,
}, OUTPUT_DIR / "evaluation_results.pt")

In [ ]:
# BƯỚC 11: VẼ CONFUSION MATRIX
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"{MODEL_NAME.upper()} — Confusion Matrix (Accuracy: {accuracy:.4f})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()
print(f"Confusion matrix saved!")

In [ ]:
# BƯỚC 12: VẼ TRAINING CURVES
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"],   label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history["train_acc"], label="Train Acc")
axes[1].plot(history["val_acc"],   label="Val Acc")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves"); axes[1].legend(); axes[1].grid(True)

plt.suptitle(f"{MODEL_NAME.upper()} — Training Curves")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()
print(f"Training curves saved!")

In [ ]:
# BƯỚC 13: NÉN KẾT QUẢ
import zipfile

zip_path = OUTPUT_DIR / "results.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fname in ["best_model.pth", "training_history.json", "evaluation_results.pt"]:
        fpath = OUTPUT_DIR / fname
        if fpath.exists():
            mb = fpath.stat().st_size / 1e6
            zf.write(fpath, arcname=fname)
            print(f"  Added: {fname} ({mb:.1f} MB)")

zip_mb = zip_path.stat().st_size / 1e6
print(f"\nAll results zipped: {zip_path} ({zip_mb:.1f} MB)")
print("Kaggle Output panel chứa toàn bộ file tự động.")